<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [1]:
import json, math
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)               # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundacion', 'colonia', 'sur', 'tuxtla', 'gutierrez']


---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [2]:
# pip install fasttext
!pip install fasttext-wheel
import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    # TODO: devolver el vector de la palabra con ft
    return ft.get_word_vector(palabra)

# TODO: imprimir la dimension del embedding y el tamano del vocabulario
dimension = ft.get_dimension()
vocab_size = len(ft.get_words())

print(f"Dimensión del embedding: {dimension}")
print(f"Tamaño del vocabulario: {vocab_size} palabras")

Dimensión del embedding: 300
Tamaño del vocabulario: 2000000 palabras


**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [3]:
# TODO: para 'sequia', 'cafe', 'chiapas', imprimir sus 5 vecinos mas cercanos
palabras_prueba = ['sequia', 'cafe', 'chiapas']

for p in palabras_prueba:
    print(f"\n--- Vecinos más cercanos a '{p}': ---")
    vecinos = ft.get_nearest_neighbors(p, k=5)
    for similitud, vecino in vecinos:
        print(f"  {vecino}: {similitud:.4f}")


--- Vecinos más cercanos a 'sequia': ---
  sequía: 0.7464
  sequias: 0.7236
  inundacion: 0.5896
  escacez: 0.5854
  sequías: 0.5713

--- Vecinos más cercanos a 'cafe': ---
  café: 0.7897
  cafes: 0.7414
  cafe.: 0.7375
  cafe-: 0.7242
  cafesito: 0.7142

--- Vecinos más cercanos a 'chiapas': ---
  chiapas.: 0.7302
  oaxaca: 0.7262
  tuxtla: 0.7059
  michoacan: 0.6912
  veracruz: 0.6861


**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [4]:
import numpy as np

def cos_vec(a, b):
    # TODO: similitud coseno entre dos vectores numpy
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Obtener los vectores de las palabras
v_agua = vec('agua')
v_hidrico = vec('hidrico')
v_jirafa = vec('jirafa')

# TODO: comparar coseno(agua, hidrico) (esperado ALTO) vs coseno(agua, jirafa) (esperado BAJO)
sim_hidrico = cos_vec(v_agua, v_hidrico)
sim_jirafa = cos_vec(v_agua, v_jirafa)

print(f"Similitud coseno entre 'agua' e 'hídrico': {sim_hidrico:.4f}")
print(f"Similitud coseno entre 'agua' y 'jirafa': {sim_jirafa:.4f}")

Similitud coseno entre 'agua' e 'hídrico': 0.4360
Similitud coseno entre 'agua' y 'jirafa': 0.2433


_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_

**A.4** Analogías por aritmética vectorial.

In [5]:
# TODO: probar rey - hombre + mujer (ft.get_analogies) y una analogia de capitales.
# Documenten un caso que funcione y uno que falle.

print("--- Analogía clásica: rey - hombre + mujer ---")
analogia_reina = ft.get_analogies('rey', 'hombre', 'mujer')
for similitud, palabra in analogia_reina[:3]:
    print(f"  {palabra}: {similitud:.4f}")

print("\n--- Analogía de Capitales (Éxito): madrid - espana + francia ---")
analogia_capital = ft.get_analogies('madrid', 'espana', 'francia')
for similitud, palabra in analogia_capital[:3]:
    print(f"  {palabra}: {similitud:.4f}")

print("\n--- Analogía Compleja / Fallo: doctor - hombre + mujer ---")
# A veces aquí se filtran sesgos culturales o estereotipos del corpus de entrenamiento
analogia_sesgo = ft.get_analogies('doctor', 'hombre', 'mujer')
for similitud, palabra in analogia_sesgo[:3]:
    print(f"  {palabra}: {similitud:.4f}")

--- Analogía clásica: rey - hombre + mujer ---
  reina: 0.6996
  princesa: 0.6584
  reina-madre: 0.5786

--- Analogía de Capitales (Éxito): madrid - espana + francia ---
  barcelona: 0.6332
  getafe: 0.5839
  bilbao: 0.5480

--- Analogía Compleja / Fallo: doctor - hombre + mujer ---
  doctora: 0.8318
  Doctora: 0.6955
  doctoras: 0.6529


_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_

acierta cuando el resultado de las operaciones matematicas estadisticas vista en la explicacion anterior, se repite con mucha frecuencia constantemente (el resultado)

Esto porque asi funciona la logica del modelo que importamos.

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [11]:
import numpy as np

def vector_documento(tokens):
    # Obtener el vector de FastText para cada token que no esté vacío
    vectores = [vec(t) for t in tokens if t.strip()]

    # Si el documento quedó vacío tras el preprocesamiento, devolvemos un vector de ceros
    if not vectores:
        return np.zeros(ft.get_dimension())

    # Calcular el promedio de los vectores en el eje 0
    return np.mean(vectores, axis=0)

# TODO: construir EMB_DOCS = {id: vector_documento(tokens)} para todo el corpus
EMB_DOCS = {ids[i]: vector_documento(documentos[i]) for i in range(len(ids))}

print(f"Diccionario EMB_DOCS construido con {len(EMB_DOCS)} vectores de documento.")

Diccionario EMB_DOCS construido con 14 vectores de documento.


**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

### B.3.1 - Implementaciones de TF-IDF y BM25 (desde Lab 2 y Lab 3)

Las siguientes celdas son **placeholders**. Deben pegar aquí sus implementaciones completas de `buscar_tfidf` (del Lab 2) y `buscar_bm25` (del Lab 3), incluyendo todas las variables y funciones auxiliares que necesiten (ej. `idf`, `tf_matrix`, etc.) para que funcionen correctamente.

In [19]:
# TODO: Pega aquí tu implementación de buscar_tfidf del Lab 2.
# Asegúrate de incluir cualquier variable o función auxiliar que necesite (ej. tf_matrix, idf_vector)

def buscar_tfidf(consulta, k=5):
    # ESTE ES UN PLACEHOLDER. REEMPLAZA CON TU CÓDIGO REAL.
    print("ADVERTENCIA: Usando placeholder para buscar_tfidf.")
    # Ejemplo de estructura de retorno esperada:
    # return [('d01', 0.5), ('d03', 0.4), ...]

    # Para evitar errores en la demostración inicial, devolvemos un mock
    # En un caso real, esto debería calcular la similitud TF-IDF
    mock_results = []
    for i in range(k):
        doc_id = ids[i % len(ids)] # Cicla a través de los IDs disponibles
        mock_results.append((doc_id, 0.0)) # Score 0.0 para indicar que es un placeholder
    return mock_results



# TODO: Pega aquí tu implementación de buscar_bm25 del Lab 3.
# Asegúrate de incluir cualquier variable o función auxiliar que necesite (ej. bm25_model)

def buscar_bm25(consulta, k=5):
    # ESTE ES UN PLACEHOLDER. REEMPLAZA CON TU CÓDIGO REAL.
    print("ADVERTENCIA: Usando placeholder para buscar_bm25.")
    # Ejemplo de estructura de retorno esperada:
    # return [('d01', 0.7), ('d05', 0.6), ...]

    # Para evitar errores en la demostración inicial, devolvemos un mock
    mock_results = []
    for i in range(k):
        doc_id = ids[i % len(ids)]
        mock_results.append((doc_id, 0.0)) # Score 0.0 para indicar que es un placeholder
    return mock_results


In [20]:
# TODO: peguen su preprocesar() del Lab 1.
def preprocesar(texto):
    """
    REEMPLAZA ESTO con tu pipeline real del Lab 1
    (ej. pasar a minúsculas, quitar puntuación, stopwords, lemmatización, etc.)
    """
    return texto.lower().split()

def buscar_semantico(consulta, k=5):
    # 1. Preprocesar la consulta usando el pipeline del Lab 1
    tokens_consulta = preprocesar(consulta)

    # 2. Obtener el vector promedio de la consulta
    v_consulta = vector_documento(tokens_consulta)

    # 3. Calcular la similitud coseno contra cada documento en EMB_DOCS
    resultados = []
    for doc_id, v_doc in EMB_DOCS.items():
        similitud = cos_vec(v_consulta, v_doc)
        resultados.append((doc_id, similitud))

    # 4. Ordenar de mayor a menor similitud y tomar los primeros k
    resultados_ordenados = sorted(resultados, key=lambda x: x[1], reverse=True)
    return resultados_ordenados[:k]

# prueba: buscar_semantico('problemas de agua') deberia recuperar d02 (crisis hidrica)
print("Prueba de búsqueda semántica:")
print(buscar_semantico('problemas de agua', k=5))

Prueba de búsqueda semántica:
[('d13', np.float32(0.45837575)), ('d02', np.float32(0.43550652)), ('d10', np.float32(0.3383556)), ('d01', np.float32(0.3192201)), ('d04', np.float32(0.31431544))]


**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [21]:
# TODO: para 3 consultas, imprimir el top-5 de TF-IDF (Lab 2), BM25 (Lab 3) y semantico, lado a lado.
#       Marquen en cuales el semantico encuentra documentos que los otros dos no.

consultas_prueba = [
    "problemas de escasez de agua",
    "desarrollo sustentable en la agricultura",
    "crisis economica y comercio internacional"
]

for c in consultas_prueba:
    print(f"\n" + "="*75)
    print(f"CONSULTA: '{c}'")
    print(f"="*75)

    # Ejecutar las búsquedas (Ajusta los nombres de tus funciones del Lab 2 y 3 aquí)
    res_tfidf = buscar_tfidf(c)[:5]   # Formato esperado: [(id, score), ...]
    res_bm25 = buscar_bm25(c)[:5]     # Formato esperado: [(id, score), ...]
    res_sem = buscar_semantico(c, k=5)

    # Imprimir cabecera de la tabla
    print(f"{'Ranking':<8} | {'TF-IDF (Lab 2)':<18} | {'BM25 (Lab 3)':<18} | {'Semántico (Lab 5)':<18}")
    print("-" * 75)

    # Construir las filas del Top 5
    for i in range(5):
        id_tf = res_tfidf[i][0] if i < len(res_tfidf) else "-"
        id_bm = res_bm25[i][0] if i < len(res_bm25) else "-"
        id_sem, sim = res_sem[i]

        # Formatear el texto para mostrar ID y los primeros caracteres del título
        txt_tf = f"{id_tf} ({titulos.get(id_tf, '')[:8]})" if id_tf != "-" else "-"
        txt_bm = f"{id_bm} ({titulos.get(id_bm, '')[:8]})" if id_bm != "-" else "-"
        txt_sem = f"{id_sem} ({titulos.get(id_sem, '')[:8]})"

        print(f"Top {i+1:<4} | {txt_tf:<18} | {txt_bm:<18} | {txt_sem:<18} (cos: {sim:.3f})")


CONSULTA: 'problemas de escasez de agua'
ADVERTENCIA: Usando placeholder para buscar_tfidf.
ADVERTENCIA: Usando placeholder para buscar_bm25.
Ranking  | TF-IDF (Lab 2)     | BM25 (Lab 3)       | Semántico (Lab 5) 
---------------------------------------------------------------------------
Top 1    | d01 (Lluvias )     | d01 (Lluvias )     | d02 (Crisis h)     (cos: 0.382)
Top 2    | d02 (Crisis h)     | d02 (Crisis h)     | d13 (Restable)     (cos: 0.345)
Top 3    | d03 (Cafe de )     | d03 (Cafe de )     | d10 (Avanza o)     (cos: 0.281)
Top 4    | d04 (Sequia a)     | d04 (Sequia a)     | d04 (Sequia a)     (cos: 0.258)
Top 5    | d05 (Turismo )     | d05 (Turismo )     | d01 (Lluvias )     (cos: 0.250)

CONSULTA: 'desarrollo sustentable en la agricultura'
ADVERTENCIA: Usando placeholder para buscar_tfidf.
ADVERTENCIA: Usando placeholder para buscar_bm25.
Ranking  | TF-IDF (Lab 2)     | BM25 (Lab 3)       | Semántico (Lab 5) 
---------------------------------------------------------

**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?

In [23]:
import numpy as np

# =====================================================================
# EXTRAÍDO DEL LAB 3: Define aquí tus diccionarios reales antes de evaluar
# =====================================================================
queries = {
    "q01": "problemas de escasez de agua",
    "q02": "desarrollo sustentable en la agricultura",
    "q03": "crisis economica y comercio internacional"
}

qrels = {
    "q01": {"d02": 3, "d05": 1, "d07": 0},  # Formato: "id_consulta": {"id_doc": relevancia}
    "q02": {"d03": 3, "d12": 2},
    "q03": {"d01": 2, "d09": 3}
}
# =====================================================================


def calcular_dcg_at_k(ganancias_relevancia):
    """Calcula el DCG basado en una lista de valores de relevancia ordenada."""
    ganancias = np.array(ganancias_relevancia)
    if ganancias.size == 0:
        return 0.0
    descuentos = np.log2(np.arange(2, ganancias.size + 2))
    return ganancias[0] + np.sum(ganancias[1:] / descuentos[:-1])

def calcular_ndcg_at_5(recuperados, consulta_id):
    """Calcula el nDCG@5 comparando los IDs recuperados contra tus qrels."""
    relevancias_sistema = [qrels.get(consulta_id, {}).get(doc_id, 0) for doc_id in recuperados]
    relevancias_ideales = sorted(list(qrels.get(consulta_id, {}).values()), reverse=True)[:5]

    dcg = calcular_dcg_at_k(relevancias_sistema)
    idcg = calcular_dcg_at_k(relevancias_ideales)

    if idcg == 0.0:
        return 0.0

    return dcg / idcg

# Listas para acumular los resultados de cada consulta anotada
ndcgs_tfidf = []
ndcgs_bm25 = []
ndcgs_semantico = []

# Iterar sobre las consultas que tienen juicios de relevancia (qrels)
for c_id in qrels.keys():
    texto_consulta = queries[c_id]

    # 1. Recuperar los top 5 IDs de cada sistema (Verifica que estas funciones ya existan en tu sesión)
    ids_tfidf = [doc_id for doc_id, _ in buscar_tfidf(texto_consulta)[:5]]
    ids_bm25 = [doc_id for doc_id, _ in buscar_bm25(texto_consulta)[:5]]
    ids_sem = [doc_id for doc_id, _ in buscar_semantico(texto_consulta, k=5)]

    # 2. Calcular nDCG@5 para cada uno y guardarlo en las listas
    ndcgs_tfidf.append(calcular_ndcg_at_5(ids_tfidf, c_id))
    ndcgs_bm25.append(calcular_ndcg_at_5(ids_bm25, c_id))
    ndcgs_semantico.append(calcular_ndcg_at_5(ids_sem, c_id))

# 3. Imprimir el reporte final de promedios
print("==================================================")
print("MÉTRICAS FINALES DE EVALUACIÓN (nDCG@5 Medio)")
print("==================================================")
print(f"Media nDCG@5 - TF-IDF (Lab 2) : {np.mean(ndcgs_tfidf):.4f}")
print(f"Media nDCG@5 - BM25   (Lab 3) : {np.mean(ndcgs_bm25):.4f}")
print(f"Media nDCG@5 - Semántico (Lab 5): {np.mean(ndcgs_semantico):.4f}")
print("==================================================")

ADVERTENCIA: Usando placeholder para buscar_tfidf.
ADVERTENCIA: Usando placeholder para buscar_bm25.
ADVERTENCIA: Usando placeholder para buscar_tfidf.
ADVERTENCIA: Usando placeholder para buscar_bm25.
ADVERTENCIA: Usando placeholder para buscar_tfidf.
ADVERTENCIA: Usando placeholder para buscar_bm25.
MÉTRICAS FINALES DE EVALUACIÓN (nDCG@5 Medio)
Media nDCG@5 - TF-IDF (Lab 2) : 0.5454
Media nDCG@5 - BM25   (Lab 3) : 0.5454
Media nDCG@5 - Semántico (Lab 5): 0.4500


_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_

## Entregables — Lab 5
- [ ] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [ ] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [ ] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
